
# GVH Diagonal Cubic `.28.21.2.1.2` — FAST
## Exact Crossing Semisimplicity and Uniform Projector Certificate

### Unique scientific lock

The exact real-root problem is already closed by `.28.21.2.1.1`.

This notebook asks only:

\[
\boxed{
\text{Are all repeated characteristic roots semisimple, and are the directional spectral projectors uniformly controlled?}
}
\]

The target chain is

\[
\boxed{
\text{exact real spectrum}
\rightarrow
\text{crossing semisimplicity}
\rightarrow
\text{uniform projectors/diagonalizers}
\rightarrow
\text{strong hyperbolicity}.
}
\]

### Hard rule

\[
\boxed{
\texttt{STRONG\_HYPERBOLICITY\_PROVEN=True}
}
\]

is forbidden unless **all** of the following are exact:

1. \(Q_2-Q_3\) crossing semisimplicity;
2. the two cubic self-coalescences;
3. global semisimplicity of the repeated light sector \(\lambda=\pm1\);
4. uniform directional projector/diagonalizer control.

Numerical conditioning is used only as a diagnostic witness. It can never replace the exact uniform certificate.

### Scope

Fixed healthy local frozen spectral-diagonal anisotropic witness only.

Direction domain:

\[
u,v,w\ge0,\qquad u+v+w=1.
\]

No global parameter-space claim.


In [1]:

from __future__ import annotations

import sys, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import sympy as sp

PARENT_2821211 = {
    "artifact":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.1_"
        "Exact_Quintic_Coefficient_Reconstruction_and_Simplex_Root_Certificate_FAST(1).ipynb",
    "executed_size_bytes": 75080,
    "executed_sha256": '8b3fbb04abd65e65fac49ce5a366ed89225254644415df7008fdb1ba9e454bbe',
    "source_exact": True,
    "exact_global_quintic_identity_materialized": True,
    "exact_quintic_factor_2x3_pass": True,
    "exact_simplex_real_root_certificate_materialized": True,
    "q2_q3_resultant_zero_locus_classified": True,
    "residual_unit_sector_separated": True,
    "global_crossing_semisimplicity_certified": False,
    "uniform_directional_projector_control_certified": False,
    "light_sector_global_semisimplicity_certified": False,
    "strong_hyperbolicity_proven": False,
}

G2821212_PROVENANCE_GATE_PASS = all([
    PARENT_2821211["source_exact"],
    PARENT_2821211["exact_global_quintic_identity_materialized"],
    PARENT_2821211["exact_quintic_factor_2x3_pass"],
    PARENT_2821211["exact_simplex_real_root_certificate_materialized"],
    PARENT_2821211["q2_q3_resultant_zero_locus_classified"],
    PARENT_2821211["residual_unit_sector_separated"],
    not PARENT_2821211["strong_hyperbolicity_proven"],
])

assert G2821212_PROVENANCE_GATE_PASS

print("Python =",sys.version.split()[0])
print("NumPy =",np.__version__)
print("SymPy =",sp.__version__)
print("G2821212_PROVENANCE_GATE_PASS =",G2821212_PROVENANCE_GATE_PASS)


Python = 3.13.15
NumPy = 2.1.3
SymPy = 1.14.0
G2821212_PROVENANCE_GATE_PASS = True



# 1. Principal-symbol reconstruction inherited verbatim

The following cells are copied from the audited `.28.21.2.1.1` source construction.

They rebuild the same:

\[
A_{\rm phys}(\mathbf n)
\]

from the same pure-GVH-P frozen witness and the same Hamilton-Dirac physical reduction.

No new action, gauge convention, or reduction is introduced here.


In [2]:

eta=sp.diag(-1,1,1,1)

a0,a1,a2=sp.symbols("a0 a1 a2",real=True)
a3=-a0-a1-a2

Abar=sp.diag(a0,a1,a2,a3)
Qbar=sp.factor(sp.trace(Abar*Abar))

KS,kappaD,Mpl2=sp.symbols(
    "K_S kappa_D Mpl2",
    real=True
)

names=[
    "n","beta1","beta2","beta3",
    "h11","h22","h33","h12","h13","h23",
    "D00","D01","D02","D03",
    "D11","D22","D33","D12","D13","D23",
]

h_basis=[]
D_basis=[]

for name in names:
    h=sp.zeros(4)
    d=sp.zeros(4)

    if name=="n":
        h[0,0]=-2
    elif name.startswith("beta"):
        i=int(name[-1])
        h[0,i]=h[i,0]=1
    elif name.startswith("h"):
        ij=name[1:]
        i,j=int(ij[0]),int(ij[1])
        h[i,j]=h[j,i]=1
    elif name.startswith("D"):
        ij=name[1:]
        i,j=int(ij[0]),int(ij[1])
        d[i,j]=d[j,i]=1

    h_basis.append(h)
    D_basis.append(d)

dA_basis=[]

for h,d in zip(h_basis,D_basis):
    dM=-eta*h*Abar+eta*d
    dA=dM-sp.trace(dM)*sp.eye(4)/4
    dA_basis.append(dA)

def build_principal_matrix(p):
    H=sp.zeros(20)

    # Pure-GVH-P sector
    for alpha_idx in range(4):
        B=[]
        J=[]
        C=[]

        for h,dA in zip(h_basis,dA_basis):
            Gamma=sp.zeros(4)

            for mu in range(4):
                for nu in range(4):
                    acc=0
                    for rho in range(4):
                        acc += eta[mu,rho]*(
                            p[alpha_idx]*h[rho,nu]
                            +p[nu]*h[rho,alpha_idx]
                            -p[rho]*h[alpha_idx,nu]
                        )/2
                    Gamma[mu,nu]=acc

            Bj=p[alpha_idx]*dA+Gamma*Abar-Abar*Gamma
            B.append(Bj)
            J.append(sp.trace(Abar*Bj))
            C.append(Abar*Bj-Bj*Abar)

        sign=eta[alpha_idx,alpha_idx]

        for j in range(20):
            for k in range(j,20):
                shape=Qbar*sp.trace(B[j]*B[k])-J[j]*J[k]
                angle=sp.trace(C[j]*C[k])

                val=(
                    -KS*sign*shape
                    -kappaD*sp.Rational(1,2)*sign*angle
                )

                H[j,k]+=val
                if k!=j:
                    H[k,j]+=val

    # Einstein-Hilbert / Fierz-Pauli benchmark
    pvec=sp.Matrix(p)
    pup=eta*pvec
    p2=(pvec.T*eta*pvec)[0]

    attrs=[]

    for h in h_basis[:10]:
        hup=eta*h*eta
        v=[
            sum(p[mu]*hup[mu,nu] for mu in range(4))
            for nu in range(4)
        ]
        w=[
            sum(pup[lam]*h[lam,nu] for lam in range(4))
            for nu in range(4)
        ]
        trh=sp.trace(eta*h)
        vp=sum(v[nu]*p[nu] for nu in range(4))
        attrs.append((h,hup,v,w,trh,vp))

    for j in range(10):
        hj,hjup,vj,wj,trj,vpj=attrs[j]

        for k in range(j,10):
            hk,hkup,vk,wk,trk,vpk=attrs[k]

            inner=sum(
                hj[mu,nu]*hkup[mu,nu]
                for mu in range(4)
                for nu in range(4)
            )

            BF=(
                p2*inner
                -sum(
                    vj[nu]*wk[nu]+vk[nu]*wj[nu]
                    for nu in range(4)
                )
                +vpj*trk
                +vpk*trj
                -p2*trj*trk
            )

            val=-Mpl2*sp.Rational(1,4)*BF

            H[j,k]+=val
            if k!=j:
                H[k,j]+=val

    return H

e0=(1,0,0,0)
e1=(0,1,0,0)
e2=(0,0,1,0)
e3=(0,0,0,1)

P_e0=build_principal_matrix(e0)
P_e1=build_principal_matrix(e1)
P_e2=build_principal_matrix(e2)
P_e3=build_principal_matrix(e3)

K_raw=P_e0

M_raw={
    1:build_principal_matrix((1,1,0,0))-P_e0-P_e1,
    2:build_principal_matrix((1,0,1,0))-P_e0-P_e2,
    3:build_principal_matrix((1,0,0,1))-P_e0-P_e3,
}

G_raw={
    (1,1):P_e1,
    (2,2):P_e2,
    (3,3):P_e3,
    (1,2):(build_principal_matrix((0,1,1,0))-P_e1-P_e2)/2,
    (1,3):(build_principal_matrix((0,1,0,1))-P_e1-P_e3)/2,
    (2,3):(build_principal_matrix((0,0,1,1))-P_e2-P_e3)/2,
}

G2821161_RAW_PENCIL_RECONSTRUCTED=all([
    K_raw.shape==(20,20),
    all(M_raw[i].shape==(20,20) for i in (1,2,3)),
    all(G_raw[key].shape==(20,20) for key in G_raw),
])

assert G2821161_RAW_PENCIL_RECONSTRUCTED

print("G2821161_RAW_PENCIL_RECONSTRUCTED =",G2821161_RAW_PENCIL_RECONSTRUCTED)


G2821161_RAW_PENCIL_RECONSTRUCTED = True


In [3]:
D_raw={i:sp.simplify(M_raw[i]/2) for i in (1,2,3)}
assert all(sp.simplify(D_raw[i]+D_raw[i].T-M_raw[i])==sp.zeros(20) for i in (1,2,3))
print("Principal Legendre representative D_i=M_i/2 fixed")

Principal Legendre representative D_i=M_i/2 fixed


In [4]:

healthy_subs={
    a0:sp.Rational(3,4),
    a1:-sp.Rational(1,5),
    a2:-sp.Rational(1,4),
    KS:1,
    kappaD:2,
    Mpl2:1,
}

K_h=K_raw.subs(healthy_subs)
M_h={i:M_raw[i].subs(healthy_subs) for i in (1,2,3)}
D_h={i:D_raw[i].subs(healthy_subs) for i in (1,2,3)}
G_h={key:G_raw[key].subs(healthy_subs) for key in G_raw}

R_kin=sp.Matrix.hstack(*K_h.columnspace())
K14=sp.simplify(R_kin.T*K_h*R_kin)
K14_inv=K14.inv()

a=[a0,a1,a2,a3]

def original_gauge_vectors(p):
    vectors=[]

    for sigma in range(4):
        zeta=[0,0,0,0]
        zeta[sigma]=1

        hg=sp.zeros(4)
        dD=sp.zeros(4)

        for mu in range(4):
            for nu in range(4):
                hg[mu,nu]=(
                    p[mu]*zeta[nu]
                    +p[nu]*zeta[mu]
                )

                dD[mu,nu]=(
                    a[nu]*p[mu]*zeta[nu]
                    +a[mu]*p[nu]*zeta[mu]
                )

        vec=sp.zeros(20,1)

        vec[0]=-hg[0,0]/2
        vec[1]=hg[0,1]
        vec[2]=hg[0,2]
        vec[3]=hg[0,3]
        vec[4]=hg[1,1]
        vec[5]=hg[2,2]
        vec[6]=hg[3,3]
        vec[7]=hg[1,2]
        vec[8]=hg[1,3]
        vec[9]=hg[2,3]

        vals=[
            dD[0,0],dD[0,1],dD[0,2],dD[0,3],
            dD[1,1],dD[2,2],dD[3,3],
            dD[1,2],dD[1,3],dD[2,3],
        ]

        for j,val in enumerate(vals,start=10):
            vec[j]=val

        vectors.append(vec)

    trace=sp.zeros(20,1)
    trace[10]=-1
    trace[14]=1
    trace[15]=1
    trace[16]=1

    vectors.append(trace)

    return vectors

N_diff=sp.Matrix.hstack(
    *original_gauge_vectors((1,0,0,0))[:4]
).subs(healthy_subs)

N_trace=sp.zeros(20,1)
N_trace[10]=-1
N_trace[14]=1
N_trace[15]=1
N_trace[16]=1

N_radial=sp.zeros(20,1)
N_radial[10]=-a0
N_radial[14]=a1
N_radial[15]=a2
N_radial[16]=a3
N_radial=N_radial.subs(healthy_subs)

N6=sp.Matrix.hstack(
    N_diff,
    N_trace,
    N_radial,
)

T20=sp.Matrix.hstack(
    R_kin,
    N6,
)

G2821161_ORIGINAL_NULL_BASIS_PASS=all([
    K_h.rank()==14,
    R_kin.shape==(20,14),
    R_kin.rank()==14,
    N6.shape==(20,6),
    N6.rank()==6,
    K_h*N6==sp.zeros(20,6),
    T20.rank()==20,
])

assert G2821161_ORIGINAL_NULL_BASIS_PASS

print("rank K_h =",K_h.rank())
print("rank N6 =",N6.rank())
print("rank T20 =",T20.rank())
print("G2821161_ORIGINAL_NULL_BASIS_PASS =",G2821161_ORIGINAL_NULL_BASIS_PASS)


rank K_h = 14
rank N6 = 6
rank T20 = 20
G2821161_ORIGINAL_NULL_BASIS_PASS = True


In [5]:
n1,n2,n3=sp.symbols("n1 n2 n3",real=True)

B_symbolic=(
    n1*M_h[1]
    +n2*M_h[2]
    +n3*M_h[3]
)

C_symbolic=(
    n1**2*G_h[(1,1)]
    +n2**2*G_h[(2,2)]
    +n3**2*G_h[(3,3)]
    +2*n1*n2*G_h[(1,2)]
    +2*n1*n3*G_h[(1,3)]
    +2*n2*n3*G_h[(2,3)]
)

G0_symbolic=sp.Matrix.hstack(
    *original_gauge_vectors((0,n1,n2,n3))[:4]
).subs(healthy_subs)

G1_symbolic=N_diff

noether_checks=[
    sp.simplify(K_h*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(K_h*G0_symbolic+B_symbolic*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(B_symbolic*G0_symbolic+C_symbolic*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(C_symbolic*G0_symbolic)==sp.zeros(20,4),
]

G2821162_PRINCIPAL_NOETHER_CHAIN_PASS=all(noether_checks)
assert G2821162_PRINCIPAL_NOETHER_CHAIN_PASS

T20_inv=T20.inv()
K14_inv=K14.inv()

print("Noether checks =",noether_checks)

Noether checks = [True, True, True, True]


In [6]:
Q_symbolic=sp.simplify(
    (T20_inv*G0_symbolic)[:14,:]
)
F_global_symbolic=sp.simplify(Q_symbolic.T)
Gram_global=sp.simplify(Q_symbolic.T*Q_symbolic)
det_Gram=sp.factor(Gram_global.det())

poly_Gram=sp.Poly(sp.expand(det_Gram),n1,n2,n3)
gram_terms=poly_Gram.terms()

gram_even_exponents=all(
    all(e%2==0 for e in monom)
    for monom,coeff in gram_terms
)
gram_positive_coeffs=all(
    bool(coeff>0)
    for monom,coeff in gram_terms
)

G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS=all([
    Q_symbolic.shape==(14,4),
    gram_even_exponents,
    gram_positive_coeffs,
    len(gram_terms)>0,
])

assert G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS

print("det Gram total degree =",poly_Gram.total_degree())
print("det Gram term count =",len(gram_terms))
print("all exponents even =",gram_even_exponents)
print("all coefficients positive =",gram_positive_coeffs)
print(
    "G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS =",
    G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS
)

det Gram total degree = 8
det Gram term count = 15
all exponents even = True
all coefficients positive = True
G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS = True


In [7]:
def direction_blocks(direction):
    x,y,z=direction

    B=(x*M_h[1]+y*M_h[2]+z*M_h[3])
    D=B/2

    C=(
        x*x*G_h[(1,1)]
        +y*y*G_h[(2,2)]
        +z*z*G_h[(3,3)]
        +2*x*y*G_h[(1,2)]
        +2*x*z*G_h[(1,3)]
        +2*y*z*G_h[(2,3)]
    )

    return {
        "B":B,"D":D,"C":C,
        "Baa":sp.simplify(R_kin.T*B*R_kin),
        "BaN":sp.simplify(R_kin.T*B*N_diff),
        "Daa":sp.simplify(R_kin.T*D*R_kin),
        "Dau":sp.simplify(R_kin.T*D*N_diff),
        "Caa":sp.simplify(R_kin.T*C*R_kin),
        "CaN":sp.simplify(R_kin.T*C*N_diff),
        "BNR":sp.simplify(N_diff.T*B*R_kin),
        "BNN":sp.simplify(N_diff.T*B*N_diff),
        "CNR":sp.simplify(N_diff.T*C*R_kin),
        "CNN":sp.simplify(N_diff.T*C*N_diff),
    }

blk_symbolic=direction_blocks((n1,n2,n3))

U_global_symbolic=sp.simplify(
    F_global_symbolic*K14_inv*blk_symbolic["Dau"]
)
Lambda_global_symbolic=sp.simplify(
    F_global_symbolic*K14_inv*blk_symbolic["BaN"]
)

det_U_global=sp.factor(U_global_symbolic.det())
det_Lambda_global=sp.factor(Lambda_global_symbolic.det())

G2821162_GLOBAL_GAUGE_MULTIPLIER_CLOSURE_PASS=all([
    sp.simplify(det_U_global-det_Gram/16)==0,
    sp.simplify(det_Lambda_global-det_Gram)==0,
])

assert G2821162_GLOBAL_GAUGE_MULTIPLIER_CLOSURE_PASS

print("det(U_global) = det(Gram)/16 :",
      sp.simplify(det_U_global-det_Gram/16)==0)
print("det(Lambda_global) = det(Gram) :",
      sp.simplify(det_Lambda_global-det_Gram)==0)

det(U_global) = det(Gram)/16 : True
det(Lambda_global) = det(Gram) : True


In [8]:
def global_gauge_F(direction):
    x,y,z=direction
    return sp.simplify(
        F_global_symbolic.subs({n1:x,n2:y,n3:z})
    )


_bridge_cache={}

def global_hamilton_dirac_bridge(direction):
    direction=tuple(sp.sympify(v) for v in direction)
    if direction in _bridge_cache:
        return _bridge_cache[direction]
    blk=direction_blocks(direction)
    F=global_gauge_F(direction)

    Umat=sp.simplify(F*K14_inv*blk["Dau"])

    Ux=sp.simplify(-Umat.inv()*F*K14_inv*blk["Daa"])
    Up=sp.simplify(Umat.inv()*F*K14_inv)

    Vx=sp.simplify(K14_inv*(-blk["Daa"]-blk["Dau"]*Ux))
    Vp=sp.simplify(K14_inv*(sp.eye(14)-blk["Dau"]*Up))

    Lmat=sp.simplify(F*K14_inv*blk["BaN"])

    rhs_x=sp.simplify(
        F*K14_inv*(blk["Baa"]*Vx+blk["Caa"]+blk["CaN"]*Ux)
    )
    rhs_p=sp.simplify(
        F*K14_inv*(blk["Baa"]*Vp+blk["CaN"]*Up)
    )

    Lx=sp.simplify(-Lmat.inv()*rhs_x)
    Lp=sp.simplify(-Lmat.inv()*rhs_p)

    Pdx=sp.simplify(
        -blk["Daa"]*Vx
        -blk["Dau"]*Lx
        -blk["Caa"]
        -blk["CaN"]*Ux
    )
    Pdp=sp.simplify(
        -blk["Daa"]*Vp
        -blk["Dau"]*Lp
        -blk["CaN"]*Up
    )

    AHD=sp.Matrix.vstack(
        sp.Matrix.hstack(Vx,Vp),
        sp.Matrix.hstack(Pdx,Pdp),
    )

    Hx=sp.simplify(
        blk["BNR"]*Vx+blk["CNR"]+blk["CNN"]*Ux
    )
    Hp=sp.simplify(
        blk["BNR"]*Vp+blk["CNN"]*Up
    )

    Cphys=sp.Matrix.vstack(
        sp.Matrix.hstack(F,sp.zeros(4,14)),
        sp.Matrix.hstack(Hx,Hp),
    )

    result={
        "AHD":AHD,
        "Cphys":Cphys,
        "F":F,
        "Hp":Hp,
        "blocks":blk,
    }
    _bridge_cache[direction]=result
    return result



In [9]:
_normalized_cache={}

def normalized_direction_bundle(direction):
    direction=tuple(sp.sympify(v) for v in direction)
    if direction in _normalized_cache:
        return _normalized_cache[direction]
    r=sp.sqrt(sum(v*v for v in direction))
    assert r!=0

    obj=global_hamilton_dirac_bridge(direction)

    S=sp.diag(*([r]*14+[sp.Integer(1)]*14))
    Sinv=sp.diag(*([1/r]*14+[sp.Integer(1)]*14))

    Ahat=sp.simplify(S*obj["AHD"]*Sinv/r)
    Chat=sp.simplify(obj["Cphys"]*Sinv)

    result={
        "r":r,
        "Ahat":Ahat,
        "Chat":Chat,
    }
    _normalized_cache[direction]=result
    return result



In [10]:
def frame_from_pivots(Chat,Ahat,pivots=None):
    if pivots is None:
        pivots=tuple(Chat.rref()[1])
    else:
        pivots=tuple(sorted(pivots))

    assert len(pivots)==8
    free=[j for j in range(28) if j not in pivots]

    Cp=Chat[:,list(pivots)]
    Cf=Chat[:,free]
    assert Cp.det()!=0

    solved=sp.simplify(-Cp.inv()*Cf)

    R=sp.zeros(28,20)
    for i,row in enumerate(pivots):
        for j in range(20):
            R[row,j]=solved[i,j]
    for j,row in enumerate(free):
        R[row,j]=1

    L=sp.zeros(20,28)
    for j,row in enumerate(free):
        L[j,row]=1

    Aphys=sp.simplify(L*Ahat*R)

    checks={
        "constraint":Chat*R==sp.zeros(8,20),
        "left_inverse":L*R==sp.eye(20),
        "rank":R.rank()==20,
        "intertwining":sp.simplify(Ahat*R-R*Aphys)==sp.zeros(28,20),
    }

    assert all(checks.values())

    return {
        "R":R,
        "L":L,
        "Aphys":Aphys,
        "pivots":pivots,
        "free":tuple(free),
        "checks":checks,
    }




# 2. Exact collision loci inherited from the certified quintic

We freeze the exact residual factors:

\[
Q_5\propto F_2F_3.
\]

There are three residual collision loci requiring special treatment:

1. one \(Q_2-Q_3\) crossing on \(v=0\);
2. one cubic double-root point on \(v=0\);
3. one cubic double-root point on \(u=0\).

The quadratic factor \(F_2\) has no self-crossing because its discriminant is strictly positive on the simplex.

The residual sector never collides with \(y=1\), so the repeated light sector is spectrally isolated from \(F_2F_3\).


In [11]:

u,v,y = sp.symbols("u v y",real=True)

F2 = sp.expand(sp.sympify(
    '3575880000*u*v - 85956352000000*u*y + 85328549440000*u + 1975584303*v**2 - 46004626366567*v*y + 45235610405161*v + 98557536706400*y**2 - 294159043526233*y + 213754531196936',
    locals={"u":u,"v":v,"y":y}
))
F3 = sp.expand(sp.sympify(
    '475200000000*u**2*v + 65062400000000*u**2*y - 86129600000000*u**2 + 514487160000*u*v**2 + 74296351360000*u*v*y - 90887492080000*u*v - 204837783800000*u*y**2 + 588610947160000*u*y - 428268669800000*u + 139196650824*v**3 + 21092071871849*v**2*y - 23950046355521*v**2 - 114531345426641*v*y**2 + 327629553487984*v*y - 228922045526471*v + 152750499165134*y**3 - 699211437911961*y**2 + 1060470112941969*y - 532367422897166',
    locals={"u":u,"v":v,"y":y}
))

Disc2 = sp.factor(sp.discriminant(F2,y))
Disc3 = sp.factor(sp.discriminant(F3,y))
Res23 = sp.factor(sp.resultant(F2,F3,y))

# Exact already-certified collision parameters.
u_cross = sp.Rational(171256595517,866048000000)
y_cross = sp.Rational(26767,13600)

u_cubic = (
    -sp.Rational(284832107,268295000)
    +sp.Rational(395941,36488120000)*sp.sqrt(16205811493)
)
v_cubic = (
    -sp.Rational(150509593748759,35641420508441)
    +sp.Rational(12815600,35641420508441)*sp.sqrt(161292621824569)
)

y_cubic_v0 = (
    sp.Rational(1035103,1395134)
    +sp.Rational(133,23717278)*sp.sqrt(16205811493)
)
y_cubic_u0 = (
    sp.Rational(4021300195103,42121678782703)
    +sp.Rational(5625200,42121678782703)*sp.sqrt(161292621824569)
)

assert sp.simplify(F2.subs({u:u_cross,v:0,y:y_cross}))==0
assert sp.simplify(F3.subs({u:u_cross,v:0,y:y_cross}))==0

assert sp.simplify(F3.subs({u:u_cubic,v:0,y:y_cubic_v0}))==0
assert sp.simplify(sp.diff(F3,y).subs({u:u_cubic,v:0,y:y_cubic_v0}))==0

assert sp.simplify(F3.subs({u:0,v:v_cubic,y:y_cubic_u0}))==0
assert sp.simplify(sp.diff(F3,y).subs({u:0,v:v_cubic,y:y_cubic_u0}))==0

G2821212_COLLISION_LOCUS_PROVENANCE_PASS=True

print("u_cross =",u_cross)
print("y_cross =",y_cross)
print("u_cubic(v=0) ~",sp.N(u_cubic,16))
print("v_cubic(u=0) ~",sp.N(v_cubic,16))
print("G2821212_COLLISION_LOCUS_PROVENANCE_PASS =",G2821212_COLLISION_LOCUS_PROVENANCE_PASS)


u_cross = 171256595517/866048000000
y_cross = 26767/13600
u_cubic(v=0) ~ 0.3197460910818454
v_cubic(u=0) ~ 0.3436969939973915
G2821212_COLLISION_LOCUS_PROVENANCE_PASS = True



# 3. Exact \(Q_2-Q_3\) crossing semisimplicity

At the exact crossing,

\[
u=u_\times,\qquad v=0,\qquad w=1-u_\times,
\]

and the common residual squared speed is

\[
y_\times=\frac{26767}{13600}>0.
\]

Because the characteristic polynomial is even in \(\lambda\), the repeated \(y_\times\) corresponds to algebraic multiplicity \(2\) at each of

\[
\lambda=\pm\sqrt{y_\times}.
\]

A basis-independent test is:

\[
\boxed{
\dim\ker(A_{\rm phys}^2-y_\times I)=4.
}
\]

If true, the combined \(\pm\sqrt{y_\times}\) eigenspaces already saturate their total algebraic multiplicity, so both signs are semisimple.


In [12]:

d_cross = (
    sp.sqrt(u_cross),
    sp.Integer(0),
    sp.sqrt(1-u_cross),
)

bundle_cross = normalized_direction_bundle(d_cross)
frame_cross = frame_from_pivots(
    bundle_cross["Chat"],
    bundle_cross["Ahat"],
)
A_cross = frame_cross["Aphys"]

M_cross = sp.simplify(
    A_cross*A_cross-y_cross*sp.eye(20)
)

rank_cross = M_cross.rank()
nullity_cross = 20-rank_cross

G2821212_Q2_Q3_CROSSING_SEMISIMPLICITY_CERTIFIED = (
    nullity_cross==4
)

assert G2821212_Q2_Q3_CROSSING_SEMISIMPLICITY_CERTIFIED

print("crossing pivots =",frame_cross["pivots"])
print("rank(Aphys^2-y_cross I) =",rank_cross)
print("nullity =",nullity_cross)
print(
    "G2821212_Q2_Q3_CROSSING_SEMISIMPLICITY_CERTIFIED =",
    G2821212_Q2_Q3_CROSSING_SEMISIMPLICITY_CERTIFIED
)


crossing pivots = (0, 1, 2, 3, 4, 5, 7, 14)
rank(Aphys^2-y_cross I) = 16
nullity = 4
G2821212_Q2_Q3_CROSSING_SEMISIMPLICITY_CERTIFIED = True



# 4. Exact light-sector witnesses versus global light-sector theorem

The exact characteristic factor contains

\[
(\lambda^2-1)^5.
\]

Thus \(\lambda=+1\) and \(\lambda=-1\) each have algebraic multiplicity \(5\).

At any fixed direction, semisimplicity is exactly equivalent to:

\[
\dim\ker(A_{\rm phys}-I)=5,
\qquad
\dim\ker(A_{\rm phys}+I)=5.
\]

This notebook tests several exact directions, including the exact \(Q_2-Q_3\) crossing.

However, a finite set of exact directions is **not** a proof over the whole simplex.

Therefore the global light-sector flag remains `False` unless a direction-global rank/minor certificate is actually materialized.


In [13]:

exact_light_directions = [
    (sp.Integer(1),sp.Integer(0),sp.Integer(0)),
    (sp.Integer(0),sp.Integer(1),sp.Integer(0)),
    (sp.Integer(0),sp.Integer(0),sp.Integer(1)),
    (sp.Integer(1),sp.Integer(2),sp.Integer(2)),
]

light_exact_ledger=[]

for d in exact_light_directions:
    norm=normalized_direction_bundle(d)
    fr=frame_from_pivots(norm["Chat"],norm["Ahat"])
    A=fr["Aphys"]

    rp=(A-sp.eye(20)).rank()
    rm=(A+sp.eye(20)).rank()

    light_exact_ledger.append({
        "direction":str(tuple(d)),
        "rank_A_minus_I":int(rp),
        "nullity_plus":int(20-rp),
        "rank_A_plus_I":int(rm),
        "nullity_minus":int(20-rm),
    })

# Include the exact algebraic Q2-Q3 crossing.
rp_cross=(A_cross-sp.eye(20)).rank()
rm_cross=(A_cross+sp.eye(20)).rank()

light_exact_ledger.append({
    "direction":"exact_Q2_Q3_crossing",
    "rank_A_minus_I":int(rp_cross),
    "nullity_plus":int(20-rp_cross),
    "rank_A_plus_I":int(rm_cross),
    "nullity_minus":int(20-rm_cross),
})

G2821212_LIGHT_SECTOR_EXACT_WITNESS_PASS = all(
    row["nullity_plus"]==5
    and row["nullity_minus"]==5
    for row in light_exact_ledger
)

G2821212_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED=False

assert G2821212_LIGHT_SECTOR_EXACT_WITNESS_PASS
assert not G2821212_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED

print(pd.DataFrame(light_exact_ledger).to_string(index=False))
print(
    "G2821212_LIGHT_SECTOR_EXACT_WITNESS_PASS =",
    G2821212_LIGHT_SECTOR_EXACT_WITNESS_PASS
)
print(
    "G2821212_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED =",
    G2821212_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED
)


           direction  rank_A_minus_I  nullity_plus  rank_A_plus_I  nullity_minus
           (1, 0, 0)              15             5             15              5
           (0, 1, 0)              15             5             15              5
           (0, 0, 1)              15             5             15              5
           (1, 2, 2)              15             5             15              5
exact_Q2_Q3_crossing              15             5             15              5
G2821212_LIGHT_SECTOR_EXACT_WITNESS_PASS = True
G2821212_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED = False



# 5. Cubic self-coalescences: exact locus, numerical eigenspace witness only

The two cubic discriminant-zero points are algebraic irrational directions.

The parent `.28.21.2.1.1` proved:

- the double-root loci are unique on their respective edges;
- triple roots are excluded;
- the cubic remains real-rooted on the whole simplex.

What is still required here is an exact matrix-rank certificate at those algebraic directions:

\[
\dim\ker(A_{\rm phys}^2-y_c I)=4.
\]

The exact principal reduction at these nested algebraic radicals is substantially more expensive than at the rational \(Q_2-Q_3\) crossing.

This FAST notebook therefore records a high-precision matrix witness **without promoting it to an exact certificate**.

That distinction is deliberate.


In [14]:

K14_np=np.array(K14.evalf(),dtype=float)
K14_inv_np=np.linalg.inv(K14_np)
R_kin_np=np.array(R_kin.evalf(),dtype=float)
N_diff_np=np.array(N_diff.evalf(),dtype=float)

M_h_np={i:np.array(M_h[i].evalf(),dtype=float) for i in (1,2,3)}
G_h_np={key:np.array(G_h[key].evalf(),dtype=float) for key in G_h}

F_global_fun=sp.lambdify((n1,n2,n3),F_global_symbolic,"numpy")

def numeric_physical_symbol(direction):
    n=np.asarray(direction,dtype=float)
    r=float(np.linalg.norm(n))
    if not (r>0):
        raise ValueError("direction must be nonzero")

    x,yv,z=n/r

    B=x*M_h_np[1]+yv*M_h_np[2]+z*M_h_np[3]
    D=B/2.0
    C=(
        x*x*G_h_np[(1,1)]
        +yv*yv*G_h_np[(2,2)]
        +z*z*G_h_np[(3,3)]
        +2*x*yv*G_h_np[(1,2)]
        +2*x*z*G_h_np[(1,3)]
        +2*yv*z*G_h_np[(2,3)]
    )

    Baa=R_kin_np.T@B@R_kin_np
    BaN=R_kin_np.T@B@N_diff_np
    Daa=R_kin_np.T@D@R_kin_np
    Dau=R_kin_np.T@D@N_diff_np
    Caa=R_kin_np.T@C@R_kin_np
    CaN=R_kin_np.T@C@N_diff_np
    BNR=N_diff_np.T@B@R_kin_np
    CNR=N_diff_np.T@C@R_kin_np
    CNN=N_diff_np.T@C@N_diff_np

    F=np.array(F_global_fun(x,yv,z),dtype=float)

    Umat=F@K14_inv_np@Dau
    Ux=-np.linalg.solve(Umat,F@K14_inv_np@Daa)
    Up=np.linalg.solve(Umat,F@K14_inv_np)

    Vx=K14_inv_np@(-Daa-Dau@Ux)
    Vp=K14_inv_np@(np.eye(14)-Dau@Up)

    Lmat=F@K14_inv_np@BaN
    rhs_x=F@K14_inv_np@(Baa@Vx+Caa+CaN@Ux)
    rhs_p=F@K14_inv_np@(Baa@Vp+CaN@Up)

    Lx=-np.linalg.solve(Lmat,rhs_x)
    Lp=-np.linalg.solve(Lmat,rhs_p)

    Pdx=-Daa@Vx-Dau@Lx-Caa-CaN@Ux
    Pdp=-Daa@Vp-Dau@Lp-CaN@Up

    AHD=np.block([[Vx,Vp],[Pdx,Pdp]])

    Hx=BNR@Vx+CNR+CNN@Ux
    Hp=BNR@Vp+CNN@Up

    Cphys=np.block([
        [F,np.zeros((4,14))],
        [Hx,Hp],
    ])

    U,svals,Vh=np.linalg.svd(Cphys,full_matrices=True)
    R=Vh[8:,:].T
    Aphys=R.T@AHD@R

    return Aphys

def numerical_nullity(M,relative_tol=1e-9):
    svals=np.linalg.svd(M,compute_uv=False)
    tol=relative_tol*max(1.0,float(svals[0]))
    return int(np.sum(svals<tol)),svals


In [15]:

collision_numeric_ledger=[]

for label,uu,vv,yy in [
    ("cubic_v0",u_cubic,sp.Integer(0),y_cubic_v0),
    ("cubic_u0",sp.Integer(0),v_cubic,y_cubic_u0),
]:
    uf=float(sp.N(uu,30))
    vf=float(sp.N(vv,30))
    wf=max(0.0,1.0-uf-vf)

    d=np.array([
        math.sqrt(max(0.0,uf)),
        math.sqrt(max(0.0,vf)),
        math.sqrt(max(0.0,wf)),
    ])

    A=numeric_physical_symbol(d)
    yf=float(sp.N(yy,30))

    nullity,svals=numerical_nullity(
        A@A-yf*np.eye(20),
        relative_tol=1e-10,
    )

    collision_numeric_ledger.append({
        "locus":label,
        "u":uf,
        "v":vf,
        "y_double":yf,
        "numeric_nullity_A2_minus_yI":nullity,
        "fifth_smallest_singular_value":
            float(np.sort(svals)[4]),
    })

G2821212_CUBIC_SELF_CROSSING_NUMERIC_SEMISIMPLICITY_WITNESS = all(
    row["numeric_nullity_A2_minus_yI"]==4
    and row["fifth_smallest_singular_value"]>1e-6
    for row in collision_numeric_ledger
)

G2821212_CUBIC_SELF_CROSSING_SEMISIMPLICITY_CERTIFIED=False

assert G2821212_CUBIC_SELF_CROSSING_NUMERIC_SEMISIMPLICITY_WITNESS
assert not G2821212_CUBIC_SELF_CROSSING_SEMISIMPLICITY_CERTIFIED

print(pd.DataFrame(collision_numeric_ledger).to_string(index=False))
print(
    "G2821212_CUBIC_SELF_CROSSING_NUMERIC_SEMISIMPLICITY_WITNESS =",
    G2821212_CUBIC_SELF_CROSSING_NUMERIC_SEMISIMPLICITY_WITNESS
)
print(
    "G2821212_CUBIC_SELF_CROSSING_SEMISIMPLICITY_CERTIFIED =",
    G2821212_CUBIC_SELF_CROSSING_SEMISIMPLICITY_CERTIFIED
)


   locus        u        v  y_double  numeric_nullity_A2_minus_yI  fifth_smallest_singular_value
cubic_v0 0.319746 0.000000  1.455813                            4                       0.160817
cubic_u0 0.000000 0.343697  1.791522                            4                       0.117175
G2821212_CUBIC_SELF_CROSSING_NUMERIC_SEMISIMPLICITY_WITNESS = True
G2821212_CUBIC_SELF_CROSSING_SEMISIMPLICITY_CERTIFIED = False



# 6. Local diagonalizer/projector conditioning diagnostic

Exact semisimplicity at a crossing is necessary but not sufficient for strong hyperbolicity.

A polynomial matrix family can be semisimple at the collision while its individual spectral projectors diverge when approaching the collision.

Therefore one must control a directional diagonalizer:

\[
A_{\rm phys}(\mathbf n)
=
S(\mathbf n)\Lambda(\mathbf n)S(\mathbf n)^{-1}
\]

with:

\[
\boxed{
\sup_{\mathbf n}
\|S(\mathbf n)\|
\|S(\mathbf n)^{-1}\|
<\infty.
}
\]

The following finite local probes are diagnostic only.

**CORRECTED IMPLEMENTATION:** the diagnostic now uses real SVD-based subspace/eigenvector constructions instead of taking `np.real` of `np.linalg.eig` eigenvectors.  Its Boolean result is recorded but never asserted, because it is not an exact certificate.

They test whether there is an obvious numerical blow-up near the three exact residual collision loci.

Even a perfectly bounded table does **not** certify the required supremum.


In [16]:

def _svd_null_basis_real(M, expected_nullity=None, relative_tol=1e-9):
    """
    Real SVD null-space basis.
    Numerical diagnostic only; never promoted to an exact certificate.
    """
    M=np.asarray(M,dtype=float)
    U,svals,Vh=np.linalg.svd(M,full_matrices=True)

    scale=max(1.0,float(svals[0]) if len(svals) else 1.0)
    eps_floor=100.0*np.finfo(float).eps*scale
    tol=max(relative_tol*scale,eps_floor)

    if expected_nullity is None:
        mask=svals<tol
        basis=Vh[mask,:].T
    else:
        if expected_nullity<=0:
            basis=np.zeros((M.shape[1],0),dtype=float)
        else:
            basis=Vh[-expected_nullity:,:].T

    if basis.size:
        basis,_=np.linalg.qr(basis)

    return basis,svals,tol


def _real_residual_eigenvector_by_svd(A,lam):
    """
    Real approximate eigenvector from the smallest right singular vector of
    A-lam I.  Avoids arbitrary complex phases/bases from eig near degeneracies.
    """
    M=np.asarray(A-float(lam)*np.eye(A.shape[0]),dtype=float)
    U,svals,Vh=np.linalg.svd(M,full_matrices=True)
    vec=Vh[-1,:].astype(float)

    nrm=float(np.linalg.norm(vec))
    if not (nrm>0.0):
        raise RuntimeError("zero residual eigenvector in numerical diagnostic")

    vec=vec/nrm

    j=int(np.argmax(np.abs(vec)))
    if vec[j]<0:
        vec=-vec

    return vec,float(svals[-1])


def stabilized_diagonalizer_condition(A):
    """
    Version-robust finite numerical diagnostic only.

    Light eigenspaces ±1:
        real SVD bases with expected dimension 5.

    Residual modes:
        eigenvalues from eigvals, but real eigenvectors reconstructed with SVD
        of A-lambda I, so arbitrary complex phases from eig are not used.

    The returned condition number is never an exact gate.
    """
    A=np.asarray(A,dtype=float)
    vals=np.linalg.eigvals(A)

    imag_max=float(np.max(np.abs(np.imag(vals))))
    if imag_max>1e-6:
        return {
            "condition":np.inf,
            "imag_max":imag_max,
            "residual_max_smin":np.inf,
            "status":"COMPLEX_NUMERIC_EIGENVALUE_DIAGNOSTIC",
        }

    vals=np.real(vals)

    Nminus,s_minus,tol_minus=_svd_null_basis_real(
        A+np.eye(20),expected_nullity=5
    )
    Nplus,s_plus,tol_plus=_svd_null_basis_real(
        A-np.eye(20),expected_nullity=5
    )

    remaining=list(range(len(vals)))

    for target in [-1.0]*5+[1.0]*5:
        j=min(remaining,key=lambda k:abs(vals[k]-target))
        remaining.remove(j)

    residual_vals=sorted(float(vals[k]) for k in remaining)

    if len(residual_vals)!=10:
        return {
            "condition":np.inf,
            "imag_max":imag_max,
            "residual_max_smin":np.inf,
            "status":"RESIDUAL_EIGENVALUE_COUNT_MISMATCH",
        }

    residual_vectors=[]
    residual_smins=[]

    for lam in residual_vals:
        vec,smin=_real_residual_eigenvector_by_svd(A,lam)
        residual_vectors.append(vec)
        residual_smins.append(smin)

    Vother=np.column_stack(residual_vectors)
    V=np.column_stack([Nminus,Nplus,Vother])

    try:
        cond=float(np.linalg.cond(V))
    except Exception:
        cond=np.inf

    return {
        "condition":cond,
        "imag_max":imag_max,
        "residual_max_smin":float(max(residual_smins)),
        "status":"NUMERIC_DIAGNOSTIC_COMPLETED",
    }


In [17]:

local_projector_ledger=[]

collision_points=[
    ("q2_q3",float(sp.N(u_cross,30)),0.0),
    ("cubic_v0",float(sp.N(u_cubic,30)),0.0),
    ("cubic_u0",0.0,float(sp.N(v_cubic,30))),
]

for label,u0,v0 in collision_points:
    for eps in [1e-3,1e-5,1e-7]:
        probes=[]

        if v0==0.0:
            probes.extend([
                (u0-eps,0.0),
                (u0+eps,0.0),
                (u0,eps),
            ])
        else:
            probes.extend([
                (0.0,v0-eps),
                (0.0,v0+eps),
                (eps,v0),
            ])

        for uu,vv in probes:
            if uu<0 or vv<0 or uu+vv>1:
                continue

            ww=1-uu-vv
            d=[
                math.sqrt(uu),
                math.sqrt(vv),
                math.sqrt(ww),
            ]
            A=numeric_physical_symbol(d)
            diag=stabilized_diagonalizer_condition(A)

            local_projector_ledger.append({
                "locus":label,
                "eps":eps,
                "u":uu,
                "v":vv,
                "diag_condition":float(diag["condition"]),
                "imag_max":float(diag["imag_max"]),
                "residual_max_smin":float(diag["residual_max_smin"]),
                "diagnostic_status":diag["status"],
            })

# This is only a numerical witness.  It must never gate an exact certificate.
G2821212_LOCAL_PROJECTOR_NUMERIC_BOUNDED_WITNESS = all(
    row["diagnostic_status"]=="NUMERIC_DIAGNOSTIC_COMPLETED"
    and np.isfinite(row["diag_condition"])
    and row["diag_condition"]<1e5
    for row in local_projector_ledger
)

G2821212_LOCAL_PROJECTOR_NUMERIC_DIAGNOSTIC_COMPLETED = True

summary = (
    pd.DataFrame(local_projector_ledger)
    .groupby(["locus","eps"],as_index=False)
    .agg({
        "diag_condition":"max",
        "imag_max":"max",
        "residual_max_smin":"max",
    })
)

print(summary.to_string(index=False))
print(
    "G2821212_LOCAL_PROJECTOR_NUMERIC_DIAGNOSTIC_COMPLETED =",
    G2821212_LOCAL_PROJECTOR_NUMERIC_DIAGNOSTIC_COMPLETED
)
print(
    "G2821212_LOCAL_PROJECTOR_NUMERIC_BOUNDED_WITNESS =",
    G2821212_LOCAL_PROJECTOR_NUMERIC_BOUNDED_WITNESS
)

if not G2821212_LOCAL_PROJECTOR_NUMERIC_BOUNDED_WITNESS:
    print(
        "NUMERIC DIAGNOSTIC ONLY: the finite conditioning witness did not "
        "pass in this numerical environment. No exact certificate is invalidated."
    )


   locus          eps  diag_condition     imag_max  residual_max_smin
cubic_u0 1.000000e-07    1.213294e+03 2.571095e-14       2.669579e-14
cubic_u0 1.000000e-05    1.213151e+03 5.632306e-14       2.690501e-14
cubic_u0 1.000000e-03    1.213285e+03 4.285899e-14       2.504708e-14
cubic_v0 1.000000e-07    1.489820e+02 1.303787e-14       1.118547e-15
cubic_v0 1.000000e-05    1.489888e+02 1.110168e-14       1.447569e-15
cubic_v0 1.000000e-03    1.494652e+02 1.114818e-13       1.324891e-15
   q2_q3 1.000000e-07    8.745530e+16 9.881609e-15       9.128565e-16
   q2_q3 1.000000e-05    1.923316e+02 1.349024e-14       1.011845e-15
   q2_q3 1.000000e-03    1.928351e+02 4.626414e-14       1.369206e-15
G2821212_LOCAL_PROJECTOR_NUMERIC_DIAGNOSTIC_COMPLETED = True
G2821212_LOCAL_PROJECTOR_NUMERIC_BOUNDED_WITNESS = False
NUMERIC DIAGNOSTIC ONLY: the finite conditioning witness did not pass in this numerical environment. No exact certificate is invalidated.



# 7. Exact-certificate ledger

We can now separate what is exact from what is only supported numerically.

## Exact in this notebook

\[
\boxed{
Q_2-Q_3\text{ crossing semisimplicity}
}
\]

is certified by an exact rank computation.

## Not yet exact

The two algebraic cubic self-coalescences currently have only a high-precision rank witness.

The light sector has exact semisimplicity witnesses at several directions, but not a direction-global minor/rank certificate.

The local diagonalizer/projector condition numbers remain numerical witnesses, not an analytic uniform bound.

Therefore the full certificate cannot yet be promoted.

This is a scientifically valid negative outcome for this attempted lock.


In [18]:

G2821212_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED = all([
    G2821212_Q2_Q3_CROSSING_SEMISIMPLICITY_CERTIFIED,
    G2821212_CUBIC_SELF_CROSSING_SEMISIMPLICITY_CERTIFIED,
])

G2821212_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED=False

G2821212_STRONG_HYPERBOLICITY_PROVEN = all([
    PARENT_2821211["exact_simplex_real_root_certificate_materialized"],
    G2821212_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED,
    G2821212_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED,
    G2821212_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED,
])

assert not G2821212_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED
assert not G2821212_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED
assert not G2821212_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED
assert not G2821212_STRONG_HYPERBOLICITY_PROVEN

print(
    "G2821212_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED =",
    G2821212_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED
)
print(
    "G2821212_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED =",
    G2821212_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED
)
print(
    "G2821212_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED =",
    G2821212_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED
)
print(
    "G2821212_STRONG_HYPERBOLICITY_PROVEN =",
    G2821212_STRONG_HYPERBOLICITY_PROVEN
)


G2821212_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED = False
G2821212_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED = False
G2821212_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED = False
G2821212_STRONG_HYPERBOLICITY_PROVEN = False



# 8. Precise blocker and next authorized repair

The blocker is no longer the real spectrum and no longer the \(Q_2-Q_3\) crossing.

The remaining exact tasks are:

### B1 — cubic algebraic collision ranks

At both exact algebraic edge points prove:

\[
\boxed{
\dim\ker(A_{\rm phys}^2-y_cI)=4.
}
\]

This should be done over the appropriate algebraic number fields, not by floating-point SVD.

### B2 — global light-sector rank certificate

Prove direction-globally:

\[
\boxed{
\operatorname{rank}(A_{\rm phys}-I)=15,
\qquad
\operatorname{rank}(A_{\rm phys}+I)=15.
}
\]

A finite atlas of exact nonvanishing \(15\times15\) minors is acceptable.

### B3 — local analytic projector control

At each residual collision prove that the individual spectral projectors have finite direction-uniform limits, or equivalently construct a locally bounded diagonalizer.

A finite numerical probe is not sufficient.

Only after B1+B2+B3 may:

\[
\boxed{
\texttt{STRONG\_HYPERBOLICITY\_PROVEN=True}
}
\]

be reconsidered.


In [19]:

G2821212_BLOCKERS = [
    "EXACT_CUBIC_ALGEBRAIC_COLLISION_RANKS",
    "GLOBAL_LIGHT_SECTOR_RANK15_ATLAS",
    "ANALYTIC_LOCAL_PROJECTOR_BOUNDEDNESS_AT_COLLISIONS",
]

G2821212_CERTIFICATE_ROUTE_REFINED=True

G2821212_NEXT_AUTHORIZED = (
    "same spectral branch only: exact algebraic collision-rank "
    "certificate + global light rank atlas + analytic local projector limits"
)

print("G2821212_BLOCKERS =",G2821212_BLOCKERS)
print(
    "G2821212_CERTIFICATE_ROUTE_REFINED =",
    G2821212_CERTIFICATE_ROUTE_REFINED
)
print("NEXT_AUTHORIZED =",G2821212_NEXT_AUTHORIZED)


G2821212_BLOCKERS = ['EXACT_CUBIC_ALGEBRAIC_COLLISION_RANKS', 'GLOBAL_LIGHT_SECTOR_RANK15_ATLAS', 'ANALYTIC_LOCAL_PROJECTOR_BOUNDEDNESS_AT_COLLISIONS']
G2821212_CERTIFICATE_ROUTE_REFINED = True
NEXT_AUTHORIZED = same spectral branch only: exact algebraic collision-rank certificate + global light rank atlas + analytic local projector limits



# 9. Four-level protocol

## Level 1 — GVH

Only the already-derived physical principal symbol \(A_{\rm phys}\) is used.

No new dynamics is introduced.

## Level 2 — established mathematics

Semisimplicity is tested by algebraic multiplicity versus kernel dimension.

Uniform strong hyperbolicity requires bounded diagonalizers/projectors; compactness alone is not used as a substitute for the missing local crossing bound.

## Level 3 — exact versus numerical

Exact:

- collision loci inherited;
- \(Q_2-Q_3\) crossing rank/semisimplicity;
- finite exact light-sector witnesses.

Numerical only:

- cubic collision nullity witnesses;
- local diagonalizer conditioning.

These are explicitly not promoted.

## Level 4 — units / observables

No SI scale, phenomenology, or observable is introduced.


In [20]:

ESTABLISHED_PHYSICS_USED_AS_BENCHMARK_NOT_SUBSTITUTE=True

LEVEL1_GVH_PASS=True
LEVEL2_ESTABLISHED_MATH_PASS=True
LEVEL3_SCOPE_LEDGER_PASS=True

UNIVERSAL_THEORY_SELECTED_SI_SCALE_RANK=0
NUMERICAL_SI_CALIBRATION_AUTHORIZED=False
LEVEL4_SI_LEDGER_PASS=True

FOUR_LEVEL_PROTOCOL_PASS=all([
    LEVEL1_GVH_PASS,
    LEVEL2_ESTABLISHED_MATH_PASS,
    LEVEL3_SCOPE_LEDGER_PASS,
    LEVEL4_SI_LEDGER_PASS,
])

assert FOUR_LEVEL_PROTOCOL_PASS

print("FOUR_LEVEL_PROTOCOL_PASS =",FOUR_LEVEL_PROTOCOL_PASS)


FOUR_LEVEL_PROTOCOL_PASS = True


In [21]:

verdict={
    "notebook":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2_"
        "Exact_Crossing_Semisimplicity_and_Uniform_Projector_Certificate_FAST",
    "parent_28_21_2_1_1":PARENT_2821211,
    "scope":{
        "background":
            "fixed healthy local frozen spectral-diagonal anisotropic witness",
        "direction_domain":
            "projective simplex u,v,w>=0, u+v+w=1",
        "global_parameter_space_claim":False,
    },
    "exact":{
        "q2_q3_crossing_semisimplicity_certified":
            bool(G2821212_Q2_Q3_CROSSING_SEMISIMPLICITY_CERTIFIED),
        "light_sector_exact_finite_witness_pass":
            bool(G2821212_LIGHT_SECTOR_EXACT_WITNESS_PASS),
    },
    "numeric_witness":{
        "cubic_self_crossing_semisimplicity":
            bool(G2821212_CUBIC_SELF_CROSSING_NUMERIC_SEMISIMPLICITY_WITNESS),
        "local_projector_diagnostic_completed":
            bool(G2821212_LOCAL_PROJECTOR_NUMERIC_DIAGNOSTIC_COMPLETED),
        "local_projector_bounded":
            bool(G2821212_LOCAL_PROJECTOR_NUMERIC_BOUNDED_WITNESS),
        "local_projector_witness_is_exact_gate":False,
    },
    "locks":{
        "cubic_self_crossing_semisimplicity_certified":
            False,
        "global_crossing_semisimplicity_certified":
            bool(G2821212_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED),
        "light_sector_global_semisimplicity_certified":
            bool(G2821212_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED),
        "uniform_directional_projector_control_certified":
            bool(G2821212_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED),
        "strong_hyperbolicity_proven":
            bool(G2821212_STRONG_HYPERBOLICITY_PROVEN),
    },
    "blockers":G2821212_BLOCKERS,
    "protocol":{
        "four_level_protocol_pass":bool(FOUR_LEVEL_PROTOCOL_PASS),
        "universal_theory_selected_SI_scale_rank":0,
    },
    "status":
        "PARTIAL_PASS_EXACT_Q2Q3_SEMISIMPLICITY_"
        "UNIFORM_PROJECTOR_CERTIFICATE_OPEN",
    "next_authorized":G2821212_NEXT_AUTHORIZED,
}

candidate_colab = Path("/content/gvh_exports")
candidate_local = Path("/mnt/data/gvh_exports_2821212")

try:
    candidate_colab.mkdir(parents=True,exist_ok=True)
    test_path = candidate_colab / ".write_test"
    test_path.write_text("ok",encoding="utf-8")
    test_path.unlink()
    export_dir = candidate_colab
except Exception:
    export_dir = candidate_local
    export_dir.mkdir(parents=True,exist_ok=True)

verdict_path=export_dir / (
    "gvh_0.3.2.7.3.7.3.3.28.21.2.1.2_"
    "Crossing_Semisimplicity_Uniform_Projector_FAST.json"
)

verdict_path.write_text(
    json.dumps(verdict,indent=2,ensure_ascii=False),
    encoding="utf-8",
)

print("STATUS =",verdict["status"])
print(
    "Q2_Q3_CROSSING_SEMISIMPLICITY_CERTIFIED =",
    verdict["exact"]["q2_q3_crossing_semisimplicity_certified"]
)
print(
    "GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED =",
    verdict["locks"]["global_crossing_semisimplicity_certified"]
)
print(
    "LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED =",
    verdict["locks"]["light_sector_global_semisimplicity_certified"]
)
print(
    "UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED =",
    verdict["locks"]["uniform_directional_projector_control_certified"]
)
print(
    "STRONG_HYPERBOLICITY_PROVEN =",
    verdict["locks"]["strong_hyperbolicity_proven"]
)
print("verdict JSON =",verdict_path)


STATUS = PARTIAL_PASS_EXACT_Q2Q3_SEMISIMPLICITY_UNIFORM_PROJECTOR_CERTIFICATE_OPEN
Q2_Q3_CROSSING_SEMISIMPLICITY_CERTIFIED = True
GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED = False
LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED = False
UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED = False
STRONG_HYPERBOLICITY_PROVEN = False
verdict JSON = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.28.21.2.1.2_Crossing_Semisimplicity_Uniform_Projector_FAST.json
